# 3.1 — VolumeKernel3D and the Riesz Map

This notebook demonstrates `VolumeKernel3D` — the primary linear functional in
`pygeoinf3d` — and the Riesz duality pairing for `Lebesgue3D`.

A **VolumeKernel3D** implements the inner-product functional
$$\ell_k[f] = \int_\Omega k(\mathbf{x})\, f(\mathbf{x})\, \mathrm{d}V$$
where $k \in L^2(\Omega)$ is the kernel and $f$ is a primal function.

**Sections:** 1. Imports, 2. VolumeKernel3D, 3. Evaluating functionals,
4. Linearity, 5. SOLA-style grid, 6. Riesz map, 7. Kernel visualisation, 8. Summary


## 1 — Imports and Setup


In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose

from pygeoinf3d import BoxDomain, Function3D, Lebesgue3D, VolumeKernel3D
from pygeoinf3d.plot import plot_function3d_axis_slice

box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)
print('BoxDomain:', box)
print('Volume  :', box.volume)
print('Lebesgue3D.dim (basis-free):', space.dim)


BoxDomain: BoxDomain(low=[0, 0, 0], high=[1, 1, 1])
Volume  : 1.0
Lebesgue3D.dim (basis-free): 0


## 2 — VolumeKernel3D Construction

`VolumeKernel3D(space, kernel)` requires `kernel` to be a `Function3D`.
**Not** a raw callable — wrap it: `Function3D(box, evaluate_callable=fn)`.


In [2]:
center = np.array([0.5, 0.5, 0.5])
sigma  = 0.15

kernel_fn = Function3D(
    box,
    evaluate_callable=lambda x: np.exp(
        -np.sum((x - center)**2, axis=1) / (2 * sigma**2)
    )
)
vk = VolumeKernel3D(space, kernel_fn)

print('type(vk):', type(vk))
print('Public methods:', [m for m in dir(vk) if not m.startswith('_')])


type(vk): <class 'pygeoinf3d.spaces.forms.VolumeKernel3D'>
Public methods: ['as_linear_operator', 'components', 'copy', 'derivative', 'domain', 'from_linear_operator', 'gradient', 'has_gradient', 'has_hessian', 'has_subgradient', 'hessian', 'kernel', 'subgradient']


## 3 — Evaluating the Functional on Test Functions

$$\ell_k[1] = \int_{[0,1]^3} k(\mathbf{x})\,\mathrm{d}V$$


In [3]:
f_const = Function3D(box, evaluate_callable=lambda x: np.ones(len(x)))
f_sin   = Function3D(box, evaluate_callable=lambda x:
    np.sin(np.pi*x[:,0]) * np.sin(np.pi*x[:,1]) * np.sin(np.pi*x[:,2]))
f_poly  = Function3D(box, evaluate_callable=lambda x: x[:,0]*x[:,1]*x[:,2])

val_const = vk(f_const)
val_sin   = vk(f_sin)
val_poly  = vk(f_poly)

print(f'vk(f_const) = {val_const:.8f}')
print(f'vk(f_sin)   = {val_sin:.8f}')
print(f'vk(f_poly)  = {val_poly:.8f}')

integral_k = box.integrate_volume(
    lambda x: np.exp(-np.sum((x-center)**2,axis=1)/(2*sigma**2)))
print(f'Direct integral of k: {integral_k:.8f}')
assert_allclose(val_const, integral_k, rtol=1e-6)
print('[PASS] vk(f_const) == integral_k  (rtol=1e-6)')


vk(f_const) = 0.05302559
vk(f_sin)   = 0.03811019
vk(f_poly)  = 0.00662820
Direct integral of k: 0.05302559
[PASS] vk(f_const) == integral_k  (rtol=1e-6)


## 4 — Linearity Verification

$\ell_k[\alpha f + \beta g] = \alpha\,\ell_k[f] + \beta\,\ell_k[g]$, $\alpha=2, \beta=-1$.


In [4]:
alpha, beta = 2.0, -1.0
f_combo = Function3D(box, evaluate_callable=lambda x:
    alpha*np.ones(len(x)) + beta*np.sin(np.pi*x[:,0])*np.sin(np.pi*x[:,1])*np.sin(np.pi*x[:,2]))

lhs = vk(f_combo)
rhs = alpha * val_const + beta * val_sin
print(f'vk(combo)               = {lhs:.10f}')
print(f'alpha*vk(f)+beta*vk(g)  = {rhs:.10f}')
print(f'|diff|                  = {abs(lhs-rhs):.2e}')
assert_allclose(lhs, rhs, atol=1e-10)
print('[PASS] Linearity holds (atol=1e-10)')


vk(combo)               = 0.0679409922
alpha*vk(f)+beta*vk(g)  = 0.0679409922
|diff|                  = 8.33e-17
[PASS] Linearity holds (atol=1e-10)


## 5 — Multiple Kernels: SOLA-style 3x3 Grid

$k_i(\mathbf{x}) = \exp(-\|\mathbf{x}-\mathbf{c}_i\|^2/2\sigma^2)$,
$\mathbf{c}_i\in\{0.25,0.5,0.75\}^2\times\{0.5\}$


In [5]:
sigma_grid   = 0.12
centers_xy   = [(cx,cy) for cy in [0.25,0.5,0.75] for cx in [0.25,0.5,0.75]]
grid_centers = np.array([[cx,cy,0.5] for cx,cy in centers_xy])

def make_kernel(c):
    return VolumeKernel3D(space, Function3D(box,
        evaluate_callable=lambda x,_c=c: np.exp(-np.sum((x-_c)**2,axis=1)/(2*sigma_grid**2))))

grid_kernels = [make_kernel(c) for c in grid_centers]
vals = np.array([k(f_const) for k in grid_kernels])
print('Kernel values on f_const (row=y, col=x):')
print(vals.reshape(3,3).round(6))

fig, ax = plt.subplots(figsize=(4,3.5))
im = ax.imshow(vals.reshape(3,3), origin='lower',
               extent=[0.125,0.875,0.125,0.875], vmin=0, cmap='inferno')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('$\\int k_i\\,\\mathrm{d}V$ for 3x3 kernel grid')
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig('/tmp/sola_grid.png', dpi=80); plt.close()
print('Saved /tmp/sola_grid.png')


Kernel values on f_const (row=y, col=x):
[[0.026247 0.026726 0.026247]
 [0.026726 0.027213 0.026726]
 [0.026247 0.026726 0.026247]]


Saved /tmp/sola_grid.png


In [6]:
f_kernel_sum = Function3D(box, evaluate_callable=lambda x:
    sum(np.exp(-np.sum((x-c)**2,axis=1)/(2*sigma_grid**2)) for c in grid_centers))

fig, ax = plt.subplots(figsize=(5,4))
plot_function3d_axis_slice(f_kernel_sum, axis='z', coord=0.5, resolution=120,
    show_plot=False, title='Sum of 9 Gaussian kernels -- slice z=0.5', ax=ax)
plt.tight_layout()
plt.savefig('/tmp/kernel_sum_slice.png', dpi=80); plt.close()
print('Saved /tmp/kernel_sum_slice.png')


Saved /tmp/kernel_sum_slice.png


## 6 — Riesz Map and Duality Pairing

| Direction | Method | Returns |
|-----------|--------|---------|
| Primal to Dual | `space.to_dual(f)` | `VolumeKernel3D` |
| Dual to Primal | `space.riesz(vk)` | `Function3D` |


In [7]:
dual_sin = space.to_dual(f_sin)
print('space.to_dual(f_sin) ->', type(dual_sin))

val_direct = dual_sin(f_const)
val_cross  = space.inner_product(f_sin, f_const)
print(f'dual_sin(f_const)      = {val_direct:.8f}')
print(f'<f_sin, f_const>_L2    = {val_cross:.8f}')
assert_allclose(val_direct, val_cross, rtol=1e-6)
print('[PASS] Duality pairing matches L2 inner product (rtol=1e-6)')

recovered = space.riesz(dual_sin)
print(f'space.riesz(dual_sin) -> {type(recovered)}')
xs = box.sample_interior(500)
assert_allclose(recovered(xs), f_sin(xs), rtol=1e-8)
print('[PASS] Recovered function equals f_sin pointwise (rtol=1e-8)')

pairing_1 = dual_sin(kernel_fn)
pairing_2 = vk(f_sin)
print(f'dual_sin(kernel_fn) = {pairing_1:.8f}')
print(f'vk(f_sin)           = {pairing_2:.8f}')
assert_allclose(pairing_1, pairing_2, rtol=1e-6)
print('[PASS] <f_sin, k> == <k, f_sin>  (symmetry, rtol=1e-6)')


space.to_dual(f_sin) -> <class 'pygeoinf3d.spaces.forms.VolumeKernel3D'>
dual_sin(f_const)      = 0.25880944
<f_sin, f_const>_L2    = 0.25880944
[PASS] Duality pairing matches L2 inner product (rtol=1e-6)
space.riesz(dual_sin) -> <class 'pygeoinf3d.core.functions.Function3D'>
[PASS] Recovered function equals f_sin pointwise (rtol=1e-8)
dual_sin(kernel_fn) = 0.03811019
vk(f_sin)           = 0.03811019
[PASS] <f_sin, k> == <k, f_sin>  (symmetry, rtol=1e-6)


## 7 — Kernel Visualisation


In [8]:
fig, ax = plt.subplots(figsize=(5,4))
plot_function3d_axis_slice(
    kernel_fn, axis='z', coord=0.5, resolution=150,
    show_plot=False,
    title=f'Gaussian kernel k(x) at z=0.5  (sigma={sigma})',
    ax=ax,
)
plt.tight_layout()
plt.savefig('/tmp/kernel_gaussian.png', dpi=80); plt.close()
print('Saved /tmp/kernel_gaussian.png')


Saved /tmp/kernel_gaussian.png


## 8 — Summary

| Feature | Class / Method | Notes |
|---------|---------------|-------|
| Linear functional | `VolumeKernel3D(space, kernel)` | `kernel` must be `Function3D` |
| Evaluate | `vk(f)` | float, $\int k\,f\,\mathrm{d}V$ |
| Forward Riesz | `space.to_dual(f)` | primal to dual |
| Inverse Riesz | `space.riesz(vk)` | dual to primal |
| Volume integral | `box.integrate_volume(fn)` | fn: (N,3) to (N,) |
| Slice plot | `plot_function3d_axis_slice(fn, axis, coord)` | fn: Function3D |

**Next:** [4_providers_3d_demo.ipynb](4_providers_3d_demo.ipynb)
